In [6]:
import numpy as np

action_num = 2
state_num = 4
nu = 0.5
N = 4
b = 3
alpha = 0.1

rng = np.random.default_rng(seed=0)


def get_directions():
    directions = rng.standard_normal((N, action_num, state_num))
    return directions


def get_tamesu_omomi(directions, weights):
    """(順重み, 逆重み)の配列を返すよ～ 配列のサイズはNだよ～"""

    tamesu_omomis = []
    for k in range(N):
        w_pluses = weights + nu * directions[k]
        w_minuses = weights - nu * directions[k]
        tamesu_omomis.append((w_pluses, w_minuses))

    return tamesu_omomis


def normalize(states):
    x, x_dot, theta, theta_dot = states

    x_s = x / 4.8  # カート位置 [-4.8,4.8] → [-1,1]
    x_dot_s = x_dot / 10.0  # 経験的な速度範囲で正規化
    theta_s = theta / 0.41887903
    theta_dot_s = theta_dot / 10.0

    return x_s, x_dot_s, theta_s, theta_dot_s

def get_action(w, states):
    scores = w.dot(states)
    if scores[0] > scores[1]:
        return 0 # 左
    else:
        return 1 # 右

def update_weights(directions, weights, rewards):
    """
    directions： (N, action_num, state_num)
    rewards: (N, 2)
    """

    # スコアが高い上位b本を選ぶ
    top = np.argsort(rewards.max(axis=1))[::-1][:b]  # 上位 b 本の番号
    kept = rewards[top]                              # 形は (b, 2)
    sigma = kept.std()

    if sigma == 0: # 点数が変わらないなら更新しない
        return weights

    diff = kept[:, 0] - kept[:, 1]  # R⁺ − R⁻、形は (b,)
    kept_dirs = directions[top]  # 形は (b, 2, 4)、diff と同じ順番
    step = [diff[i] * kept_dirs[i] for i in range(b)]

    return weights + alpha / (b * sigma) * np.sum(step, axis=0)


def run_episode(env, w):
    observation, _ = env.reset()

    total_reward = 0

    while True:
        observation = normalize(observation) # 別に正規化しなくても余裕で解けるが、正規化したら1ステップで終わる
        a = get_action(w, observation)
        observation, reward, terminated, truncated, _ = env.step(a)
        total_reward += reward
        done = terminated or truncated
        if done:
            break

    return total_reward

In [7]:
import gymnasium as gym
import time

def evaluate(env, w, n=10):
    """重み w で n 回走らせて、点数の配列を返す"""
    return np.array([run_episode(env, w) for _ in range(n)])


env = gym.make("CartPole-v1")
env.reset(seed=1)  # 再現用。最初の1回だけ

steps = 10
eval_n = 50 # 評価用に何回走らせるか

# 重み
weights = np.zeros((action_num, state_num))

print(f"{'step':>4} | {'mean':>6} {'min':>5} {'max':>5} | {'solved':>6}")
print("-" * 38)

train_time = 0.0

for i in range(steps):
    t0 = time.perf_counter()  # ここから学習

    directions = get_directions()
    tamesu_omomis = get_tamesu_omomi(directions, weights)

    # N x 2(順、逆向き)回走らせる
    rewards = []  # (N, 2)
    for w_p, w_m in tamesu_omomis:
        r_p = run_episode(env, w_p)
        r_m = run_episode(env, w_m)
        rewards.append((r_p, r_m))

    weights = update_weights(directions, weights, np.array(rewards))

    train_time += time.perf_counter() - t0  # ここまで学習

    # 評価
    scores = evaluate(env, weights, eval_n)
    solved = (scores == 500).sum()

    print(
        f"{i+1:>4} | {scores.mean():>6.1f} {scores.min():>5.0f} {scores.max():>5.0f} | {solved:>3}/{eval_n}"
    )

    # 平均が500点=完全クリアなら終了

    if scores.mean() >=500:
        print("完全クリア！")
        break

print(f"学習時間: {train_time:.3f} 秒")

step |   mean   min   max | solved
--------------------------------------
   1 |  500.0   500   500 |  50/50
完全クリア！
学習時間: 0.011 秒


In [ ]:
# 500回走らせて平均報酬を求める

scores = evaluate(env, weights, 500)

print("500回の平均報酬:", scores.mean())

## GIFで保存

In [3]:
import gymnasium as gym
import numpy as np
from PIL import Image


def record_gif(w, path="cartpole.gif", seed=0):
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames = []

    while True:
        frames.append(Image.fromarray(env.render()))
        obs, _, terminated, truncated, _ = env.step(
            get_action(w, np.array(normalize(obs)))
        )
        if terminated or truncated:
            break

    env.close()
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=20, loop=0)


record_gif(weights)